# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Optimal Zero Pair AC Selection 
5. Zigzag Scan
6. Zero AC Pair Construction
7. Adaptive Payload Assignment
8. Preditiction Error Expansion Based On Turtle Shell Embedding
9. Stego DCT Coefficients
10. Entropy Coding
11. Stego Image

In [345]:
from PIL import Image
from performance import psnr, fsi, ssim, ssim_2
from zigzag import zigzag, inverse_zigzag
from math import ceil, floor, log2, log10, sqrt
import copy
import cv2
import jpeglib
import numpy as np
import pandas as pd
import import_ipynb
import matplotlib.pyplot as plt
import turtleShell
import FrequencyDomain as FD

In [346]:
# Global variable
scale_factor = 1.0

In [347]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            # block = block * 255.0 # coba pakai qf 35 untuk mendapatkan koefisien yang lebih besaro
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

def get_compress_quantized_coefficients(image_path):
    get_qf = image_path.find("_qf")
    target_qf = int(image_path[get_qf+3:get_qf+5])
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [348]:
def convert_data_to_bits(data):
    data = data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in data)
    lendata = len(data_bin)
    return data_bin, lendata

In [349]:
def causal_neighboor_smoothness(image_path):
    coeffs = get_quantized_coefficients(image_path)
    smoothness_block = []
    for idx in range(1, len(coeffs)):
        sum_z_k = np.sum(coeffs[idx-1][1:] == 0)
        sum_ac_k = np.sum(np.abs(coeffs[idx-1][1:]))
        smoothness_block.append(((idx), sum_z_k, sum_ac_k))
    mean_ac = np.mean([block[2] for block in smoothness_block])
    return smoothness_block, int(mean_ac)

In [350]:
def optimal_zero_pair_selection(image_path, threshold, payload=None, t_smooth=None):
    coeffs = get_quantized_coefficients(image_path) 
    num_pairs = (len(coeffs[0]) - 1) // 2
    bits_at_position = [0] * num_pairs 
    zero_pair_counts = [0] * num_pairs 
    t_select = 0
    total_ec_global = 0 

    for idx in range(1, len(coeffs)):
        sum_ac_prev = np.sum(np.abs(coeffs[idx-1][1:]))
        N = 4 if (sum_ac_prev < t_smooth) else 3
        idx_pairs = 0
        for k in range(2, 64, 2): 
            if idx_pairs >= threshold: break
            e1 = coeffs[idx][k-1] - coeffs[idx - 1][k-1]
            e2 = coeffs[idx][k] - coeffs[idx - 1][k]
            if e1 == 0 and e2 == 0:
                bits_at_position[idx_pairs] += N
                zero_pair_counts[idx_pairs] += 1
                total_ec_global += N
            idx_pairs += 1

    if payload:
        if total_ec_global < payload:
            raise ValueError(f"Warning: Kapasitas total ({total_ec_global}) tidak mencukupi payload ({payload})")    
        else:
            current_accumulated_bits = 0
            for i in range(len(bits_at_position)):
                current_accumulated_bits += bits_at_position[i]
                if current_accumulated_bits >= payload:
                    t_select = i + 1 
                    break

    print(f"Statistik Zero Pairs per Posisi Zigzag: {zero_pair_counts}")
    print(f"Kapasitas per posisi zigzag (bits): {bits_at_position}")
    print(f"Optimal Threshold T yang dipilih: {t_select}")
    print(f"Total Kapasitas tersedia: {total_ec_global} bits")
    return t_select, total_ec_global, zero_pair_counts

In [351]:
def construct_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0

    orig_coeffs = get_quantized_coefficients(image_path)
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zz = zigzag(block)
            if not np.array_equal(zz, orig_coeffs[i*num_h_blocks+j]):
                print("Block mapping mismatch at", i, j)    

    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            # block = block / 255.0 # kembalikan ke bentuk semula sebelum disimpan ke im.Y
            im.Y[i, j] = block
            idx += 1

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

def construct_compress_stego_file(image_path, new_coeffs):
    get_qf = image_path.find("_qf")
    target_qf = int(image_path[get_qf+3:get_qf+5])
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            enquantized = block.astype(np.float64) * FD.custom_q_mat(target_qf)
            im.Y[i, j] = np.round(enquantized / FD.custom_q_mat(100)).astype(np.int16)
            idx += 1

    im.qt[0] = FD.custom_q_mat(100)
    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

In [352]:
def recovered_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "recovered-images/recovered_" + image_path.split("/")[-1]
    print(f"Recovered image saved to {output_path}")
    im.write_dct(output_path)

In [353]:
def rdh_embed_process(image_path, data_bin, t_select, s_block, t_smooth):
    coeffs = get_quantized_coefficients(image_path) 
    # coeffs = get_compress_quantized_coefficients(image_path)
    orig_coeffs = copy.deepcopy(coeffs) 
    data_idx = 0
    lendata = len(data_bin)
    
    for idx in range(1, len(coeffs)):
        sum_ac_prev = np.sum(np.abs(orig_coeffs[idx-1][1:]))
        N = 4 if (sum_ac_prev < t_smooth) else 3
        mode = "8N" if N == 3 else "17N"
        radius = 1 if N == 3 else 2 
        
        for k in range(2, t_select * 2 + 1, 2):
            e1 = int(orig_coeffs[idx][k-1] - orig_coeffs[idx - 1][k-1])
            e2 = int(orig_coeffs[idx][k] - orig_coeffs[idx - 1][k])
            e1_mod, e2_mod = e1, e2
            if e1 == 0 and e2 == 0 and data_idx < lendata:
                bits = data_bin[data_idx : data_idx + N].ljust(N, '0')
                data_idx += N
                target_val = int(bits, 2)
                candidate_coords = turtleShell.get_kxk_nearest_zero(0, 0, N)
                e1_mod, e2_mod = turtleShell.find_val_from_zero(candidate_coords, target_val, (e1, e2), mode=mode)
            elif not (e1 == 0 and e2 == 0):
                e1_mod = e1 + int(np.sign(e1)) * radius if e1 != 0 else 0
                e2_mod = e2 + int(np.sign(e2)) * radius if e2 != 0 else 0

            coeffs[idx][k-1] = orig_coeffs[idx - 1][k-1] + e1_mod
            coeffs[idx][k] = orig_coeffs[idx - 1][k] + e2_mod

    construct_stego_file(image_path, coeffs)
    # construct_compress_stego_file(image_path, coeffs)
    return coeffs

In [354]:
def rdh_extract_process(stego_image_path, t_select, t_smooth):
    coeffs = get_quantized_coefficients(stego_image_path) 
    bit_stream = ""    
    secret_data = "" 
    stop_extraction = False

    for idx in range(1, len(coeffs)):
        sum_ac_k = np.sum(np.abs(coeffs[idx-1][1:]))
        N = 4 if (sum_ac_k < t_smooth) else 3
        mode = "8N" if N == 3 else "17N" 
        radius = 1 if N == 3 else 2 

        for k in range(2, t_select * 2 + 1, 2): 
            d1 = int(coeffs[idx][k-1] - coeffs[idx - 1][k-1])
            d2 = int(coeffs[idx][k] - coeffs[idx - 1][k])
            e1, e2 = d1, d2
            if turtleShell.is_in_central_shell(d1, d2, mode=mode):
                if not stop_extraction:
                    val = turtleShell.get_zero_matrix_value(d1, d2, mode=mode)
                    bit_stream += format(val, f'0{N}b')
                    while len(bit_stream) >= 8:
                        byte = bit_stream[:8]
                        bit_stream = bit_stream[8:]
                        char_val = int(byte, 2)
                        if char_val == 0: 
                            stop_extraction = True
                            break
                        secret_data += chr(char_val)
                e1, e2 = 0, 0
            else:
                e1 = d1 - int(np.sign(d1)) * radius if d1 != 0 else 0
                e2 = d2 - int(np.sign(d2)) * radius if d2 != 0 else 0

            coeffs[idx][k-1] = coeffs[idx - 1][k-1] + e1
            coeffs[idx][k] = coeffs[idx - 1][k] + e2

    # recovered_stego_file(stego_image_path, coeffs)
    return secret_data

In [355]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [356]:
# RDH with Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def rdh_encode(image_path, secret_data):
    ori_coeff = get_quantized_coefficients(image_path)
    data_bin, lendata = convert_data_to_bits(secret_data)
    print(secret_data)
    print(f"Data bits: {data_bin}")
    print(f"Data bits length: {lendata}")    
    s_block, mean_thresold = causal_neighboor_smoothness(image_path)
    t_select, total_ec, _ = optimal_zero_pair_selection(image_path, 21, payload=len(secret_data)*8, t_smooth=mean_thresold)
    coeff_after_embed = rdh_embed_process(image_path, data_bin, t_select, s_block, mean_thresold)
    diff = np.array(ori_coeff) - np.array(coeff_after_embed)
    print("Total changed coeff:", np.sum(diff != 0))
    print("Max change:", np.max(np.abs(diff)))
    return ori_coeff, coeff_after_embed, diff, t_select, mean_thresold, total_ec

In [357]:
# RDH with Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def rdh_decode(stego_image_path, t_select = 21, t_smooth=2):
    secret_data = rdh_extract_process(stego_image_path, t_select=t_select, t_smooth=t_smooth)
    return secret_data

In [358]:
pay_size = 1 * 1000
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf50.jpeg"
stego_image_path = f"stego_{cover_image_path}"
data = read_text_file(f"{payload_folder}{pay_size}bits.txt")
ori_coeffs, embedded_coeffs, diff, t_select, mean_thresold, total_ec = rdh_encode(f"{cover_folder}{cover_image_path}", data)

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. U
Data bits: 01001100011011110111001001100101011011010010000001101001011100000111001101110101011011010010000001100100011011110110110001101111011100100010000001110011011010010111010000100000011000010110110101100101011101000010110000100000011000110110111101101110011100110110010101100011011101000110010101110100011101010111001000100000011000010110010001101001011100000110100101110011011000110110100101101110011001110010000001100101011011000110100101110100001011100010000001010011011001010110010000100000011001000110111100100000011001010110100101110101011100110110110101101111011001000010000001110100011001010110110101110000011011110111001000100000011010010110111001100011011010010110010001101001011001000111010101101110011101000010000001110101011101000010000001101100011000010110001001101111011100100110010100100000011001010111010000100000011001000110111101101100011011110111001

In [359]:
secret_data = rdh_decode(f"{stego_folder}{stego_image_path}", t_select=t_select, t_smooth=mean_thresold)
print("Extracted Data:", secret_data)

Extracted Data: Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. U


In [360]:
# Test performance metrics
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value_2 = ssim_2(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")
print(f"SSIM_2: {ssim_value_2.mean()}")

Size cover: 45551
Size stego: 47609
PSNR: 33.24872103466981 dB
FSI: 2058.0
SSIM: 0.9329396994517806
SSIM_2: 0.9329396994517765


In [361]:
# Test performance metrics
recovered_folder = "recovered-images/"
recovered_image = f"recovered_{stego_image_path}"
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{recovered_image}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{recovered_image}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{recovered_folder}{recovered_image}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

OSError: image file is truncated (7 bytes not processed)

In [ ]:
# DEBUG
print("Difference Coefficients:")
for i in range(len(diff)):
    print(diff[i])

print("-" * 30)

print("Coeff Before Embedded:")
for i in range(len(ori_coeffs)):
    print(ori_coeffs[i])

print("-" * 30)

print("Coeff After Embedded:")
for i in range(len(embedded_coeffs)):
    print(embedded_coeffs[i])

Difference Coefficients:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ 0.  0. -1. -1.  1. -1.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
[ 0. -1. -1.  1. -1. -1.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
[0. 1. 0. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ 0.  1.  1.  1. -1. -1. -1.  0.  0.  0.  0.  0.  0.  0.  0. 

In [ ]:
im = jpeglib.read_dct(f"{stego_folder}{stego_image_path}")
print("DCT Coefficients from Stego Image:")
print(im.Y)
print(im.qt[0])

im2 = jpeglib.read_dct(f"{recovered_folder}{recovered_image}")
print("DCT Coefficients from Recovered Image:")
print(im2.Y)
print(im2.qt[0])

diff_coeff = np.array(im2.Y) - np.array(im.Y)
print("Difference Coefficients between Recovered and Stego Image:")
print(diff_coeff)
print("Max difference in coefficients:", np.max(np.abs(diff_coeff)))
print("Total changed coefficients between recovered and stego:", np.sum(diff_coeff != 0))
for i in range(diff_coeff.shape[0]):
    for j in range(diff_coeff.shape[1]):
        if np.any(diff_coeff[i, j] != 0):
            print(f"Block ({i}, {j}) has changes:\n{diff_coeff[i, j]}")

DCT Coefficients from Stego Image:
[[[[-384   77  -80 ...  -40   51    0]
   [ -24   36   84 ...  -58  -60   55]
   [   0  -26  -64 ...   57    0    0]
   ...
   [   0    0    0 ...    0    0    0]
   [   0    0    0 ...    0    0    0]
   [   0    0    0 ...    0    0    0]]

  [[-384   77  -29 ...  -80    0    0]
   [ -11  -25    0 ...    0    0    0]
   [ 127  -65    0 ...    0    0    0]
   ...
   [   0  -36    0 ...    0    0    0]
   [   0    0    0 ...    0    0    0]
   [   0    0    0 ...    0    0    0]]

  [[-416  111   41 ...  -40    0    0]
   [  49   85   70 ...    0    0    0]
   [  83  -26   16 ...   57    0    0]
   ...
   [ -24    0    0 ...    0    0    0]
   [   0    0    0 ...    0    0    0]
   [   0    0    0 ...    0    0    0]]

  ...

  [[ 160   54  -21 ...  -40    0    0]
   [  -1  109  -84 ...    0    0    0]
   [ 127  -39  -48 ...    0    0    0]
   ...
   [  48    0  -55 ...    0    0    0]
   [ -49  -64    0 ...    0    0    0]
   [   0    0    0 ...    0

In [ ]:
cover = np.array(Image.open(f"{cover_folder}{cover_image_path}").convert('L'), dtype=np.float64)
stego = np.array(Image.open(f"{stego_folder}{stego_image_path}").convert('L'), dtype=np.float64)

spatial_difference = np.abs(cover - stego) ** 2
mse = np.mean(spatial_difference)
psnr_value = 20 * np.log10(255.0 / np.sqrt(mse))
print(f"Spatil difference:")
for i in range(len(spatial_difference)):
    print(spatial_difference[i])
# print(f"PSNR in Spatial Domain: {psnr_value} dB")


Spatil difference:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 0. 1.
 1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0.
 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 1. 1. 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1.
 0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1.
 1. 0. 1. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0.
 0. 0. 1. 1. 0. 0. 0. 0. 1. 1. 0. 1. 0. 0. 1. 0. 1. 1. 1. 0. 0. 0. 0. 1.
 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1.
 1. 0. 1. 1. 1. 1. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.
 0. 1. 0. 1. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0.
 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 1

# Analisis Performance

In [ ]:
# custom_quality = [50,60,70,80,90]
# payload_list = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 11000, 12000, 13000, 14000, 15000]
# stego_image_folder = "stego-images/"
# cover_image_folder = "cover-images/"
# cover_image_base = ["airplane", "baboon", "boat", "lake", "peppers", "splash"]

# df = pd.DataFrame(columns=['Image', 'Quality', 'PSNR', 'SSIM', 'FSI', 'Payload', 'Max EC', 'Success'])

# for img in cover_image_base:
#     for q in custom_quality:
#         for psize in payload_list:
#             detailed_img = f"{cover_image_folder}{img}_qf{q}.jpeg"
#             print(f"Analyzing {detailed_img}")
#             data = read_text_file(f"{payload_folder}{psize}bits.txt")
#             ori_coeffs, embedded_coeffs, diff, t_select, mean_thresold, total_ec = rdh_encode(f"{detailed_img}", data)
#             psnr_value = psnr(detailed_img, f"{stego_image_folder}stego_{img}_qf{q}.jpeg")
#             ssim_value = ssim(detailed_img, f"{stego_image_folder}stego_{img}_qf{q}.jpeg")
#             fsi_value = fsi(detailed_img, f"{stego_image_folder}stego_{img}_qf{q}.jpeg")
#             secret_data = rdh_decode(f"{stego_image_folder}stego_{img}_qf{q}.jpeg", t_select=t_select, t_smooth=mean_thresold)
#             print(f"After Encoding Secret Data : {secret_data}")
#             success = "Yes" if secret_data == data else "No"
#             df.loc[len(df)] = [img, q, psnr_value, ssim_value, fsi_value, psize, total_ec, success]

# df.to_csv("proposed_rdh_results_2.csv", index=False)
# df